In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

TABLA_BRONZE = "proyecto_final.bronze.empleados"
TABLA_SILVER = "proyecto_final.silver.empleados"
TABLA_CUARENTENA = "proyecto_final.silver.cuarentena"

df_bronze = spark.table(TABLA_BRONZE)

print("=== ESQUEMA BRONZE EMPLEADOS ===")
df_bronze.printSchema()

print("\n=== TOTAL REGISTROS ===")
print(df_bronze.count())

print("\n=== MUESTRA DE REGISTROS ===")
display(df_bronze.limit(10))

In [0]:
from pyspark.sql import functions as F

df_bronze = spark.table("proyecto_final.bronze.empleados")

df_resumen = df_bronze.agg(
    F.count("*").alias("total_registros"),
    F.countDistinct("id_empleado").alias("empleados_distintos"),

    F.sum(
        F.when(
            F.col("id_empleado").isNull() |
            (F.trim(F.col("id_empleado")) == ""),
            1
        ).otherwise(0)
    ).alias("id_empleado_vacio"),

    F.sum(
        F.when(
            F.col("fecha_evento").isNull() |
            (F.trim(F.col("fecha_evento")) == ""),
            1
        ).otherwise(0)
    ).alias("fecha_evento_vacia")
)

display(df_resumen)

In [0]:
from pyspark.sql import functions as F

df_eventos = (
    df_bronze
    .groupBy("tipo_evento")
    .agg(
        F.count("*").alias("total_registros"),
        F.countDistinct("id_empleado").alias("empleados_distintos")
    )
    .orderBy(F.desc("total_registros"))
)

display(df_eventos)

In [0]:
from pyspark.sql import functions as F

df_fechas = df_bronze.select(
    "id_empleado",
    "tipo_evento",
    "fecha_evento",
    "fecha_registro",
    "archivo_origen",
    "fecha_ingesta"
)

print("=== DISTRIBUCIÓN POR ARCHIVO ===")

display(
    df_fechas
    .groupBy("archivo_origen")
    .agg(
        F.count("*").alias("total_eventos"),
        F.min("fecha_ingesta").alias("primera_ingesta"),
        F.max("fecha_ingesta").alias("ultima_ingesta")
    )
    .orderBy("archivo_origen")
)

print("=== MUESTRA DE FECHAS ===")

display(
    df_fechas
    .select("fecha_evento", "fecha_registro")
    .distinct()
    .limit(15)
)

print("=== FORMATOS Y RANGOS DE FECHAS ===")

df_fechas_validacion = df_fechas.withColumn(
    "fecha_evento_date",
    F.coalesce(
        F.try_to_timestamp("fecha_evento", F.lit("yyyy-MM-dd")).cast("date"),
        F.try_to_timestamp("fecha_evento", F.lit("dd/MM/yyyy")).cast("date")
    )
).withColumn(
    "fecha_registro_ts",
    F.coalesce(
        F.try_to_timestamp("fecha_registro", F.lit("yyyy-MM-dd HH:mm:ss")),
        F.try_to_timestamp("fecha_registro", F.lit("yyyy-MM-dd'T'HH:mm:ss"))
    )
)

display(
    df_fechas_validacion.agg(
        F.min("fecha_evento_date").alias("fecha_evento_min"),
        F.max("fecha_evento_date").alias("fecha_evento_max"),
        F.min("fecha_registro_ts").alias("fecha_registro_min"),
        F.max("fecha_registro_ts").alias("fecha_registro_max"),
        F.sum(
            F.when(F.col("fecha_evento_date").isNull(), 1).otherwise(0)
        ).alias("fecha_evento_no_convertible"),
        F.sum(
            F.when(F.col("fecha_registro_ts").isNull(), 1).otherwise(0)
        ).alias("fecha_registro_no_convertible")
    )
)

In [0]:
from pyspark.sql import functions as F

df_historia = (
    df_fechas_validacion
    .filter(F.col("fecha_evento_date").isNotNull())
)

print("=== EVENTOS DEL MISMO EMPLEADO EN EL MISMO DÍA ===")

df_mismo_dia = (
    df_historia
    .groupBy("id_empleado", "fecha_evento_date")
    .agg(
        F.count("*").alias("total_eventos"),
        F.countDistinct("fecha_registro_ts").alias("registros_distintos")
    )
    .filter(F.col("total_eventos") > 1)
)

display(
    df_mismo_dia.orderBy(
        F.desc("total_eventos"),
        "id_empleado",
        "fecha_evento_date"
    )
)

print("=== RESUMEN EVENTOS MISMO DÍA ===")

display(
    df_mismo_dia.agg(
        F.count("*").alias("grupos_mismo_dia"),
        F.sum("total_eventos").alias("eventos_involucrados"),
        F.sum(
            F.when(
                F.col("total_eventos") > F.col("registros_distintos"),
                1
            ).otherwise(0)
        ).alias("grupos_con_empate_fecha_registro")
    )
)

print("=== EVENTOS TARDÍOS ENTRE LOTES ===")

df_tardios = (
    df_historia
    .withColumn(
        "lote",
        F.regexp_extract(
            F.col("archivo_origen"),
            r"lote_(\d+)",
            1
        )
    )
    .withColumn(
        "mes_evento",
        F.date_format("fecha_evento_date", "yyyy-MM")
    )
)

display(
    df_tardios
    .groupBy("lote", "mes_evento")
    .count()
    .orderBy("lote", "mes_evento")
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_orden_llegada = (
    df_historia
    .withColumn(
        "lote_num",
        F.regexp_extract(
            F.col("archivo_origen"), r"lote_(\d+)", 1
        ).cast("int")
    )
    .withColumn(
        "orden_evento",
        F.struct(
            F.col("fecha_evento_date"),
            F.col("fecha_registro_ts")
        )
    )
)

w_llegada = (
    Window
    .partitionBy("id_empleado")
    .orderBy(
        F.col("lote_num").asc(),
        F.col("fecha_ingesta").asc(),
        F.col("fecha_registro_ts").asc(),
        F.col("fecha_evento_date").asc(),
        F.col("archivo_origen").asc()
    )
    .rowsBetween(Window.unboundedPreceding, -1)
)

df_llegadas_tardias = (
    df_orden_llegada
    .withColumn(
        "max_evento_anterior",
        F.max("orden_evento").over(w_llegada)
    )
    .withColumn(
        "es_tardio",
        F.when(
            F.col("orden_evento") < F.col("max_evento_anterior"),
            True
        ).otherwise(False)
    )
)

print("=== RESUMEN DE LLEGADAS TARDÍAS ===")

display(
    df_llegadas_tardias
    .groupBy("lote_num")
    .agg(
        F.count("*").alias("eventos_validos"),
        F.sum(
            F.col("es_tardio").cast("int")
        ).alias("eventos_tardios")
    )
    .orderBy("lote_num")
)

print("=== EJEMPLOS DE EVENTOS TARDÍOS ===")

display(
    df_llegadas_tardias
    .filter(F.col("es_tardio"))
    .select(
        "id_empleado",
        "lote_num",
        "fecha_evento_date",
        "fecha_registro_ts",
        "max_evento_anterior"
    )
    .orderBy("id_empleado", "lote_num")
    .limit(20)
)

In [0]:
from pyspark.sql import functions as F

df_salarios = (
    df_bronze
    .withColumn(
        "salario_limpio",
        F.regexp_replace(
            F.trim(F.col("salario")),
            ",",
            "."
        )
    )
    .withColumn(
        "salario_decimal",
        F.expr(
            "try_cast(salario_limpio AS DECIMAL(12,2))"
        )
    )
)

print("=== VALIDACIÓN GENERAL DE SALARIOS ===")

display(
    df_salarios.agg(
        F.count("*").alias("total_registros"),

        F.sum(
            F.when(
                F.col("salario").isNull() |
                (F.trim(F.col("salario")) == ""),
                1
            ).otherwise(0)
        ).alias("salario_vacio"),

        F.sum(
            F.when(
                F.col("salario").isNotNull() &
                (F.trim(F.col("salario")) != "") &
                F.col("salario_decimal").isNull(),
                1
            ).otherwise(0)
        ).alias("salario_no_convertible"),

        F.sum(
            F.when(
                F.col("salario_decimal") < 500,
                1
            ).otherwise(0)
        ).alias("salario_menor_500"),

        F.sum(
            F.when(
                F.col("salario_decimal") > 20000,
                1
            ).otherwise(0)
        ).alias("salario_mayor_20000"),

        F.min("salario_decimal").alias("salario_minimo"),
        F.max("salario_decimal").alias("salario_maximo")
    )
)

print("=== EJEMPLOS DE SALARIOS FUERA DE RANGO ===")

display(
    df_salarios
    .filter(
        (F.col("salario_decimal") < 500) |
        (F.col("salario_decimal") > 20000)
    )
    .select(
        "id_empleado",
        "tipo_evento",
        "salario",
        "salario_decimal",
        "fecha_evento",
        "archivo_origen"
    )
    .limit(20)
)

In [0]:
from pyspark.sql import functions as F

df_dni = df_bronze.withColumn(
    "dni_limpio",
    F.when(
        F.col("dni").isNull() |
        (F.trim(F.col("dni")) == ""),
        F.lit(None).cast("string")
    ).otherwise(F.trim(F.col("dni")))
)

print("=== VALIDACION GENERAL DNI ===")

display(
    df_dni.agg(
        F.count("*").alias("total_eventos"),
        F.sum(
            F.when(
                F.col("dni_limpio").isNull(), 1
            ).otherwise(0)
        ).alias("eventos_dni_vacio"),
        F.countDistinct("dni_limpio").alias("dni_distintos")
    )
)

print("=== DNI COMPARTIDOS ENTRE EMPLEADOS ===")

df_dni_compartidos = (
    df_dni
    .filter(F.col("dni_limpio").isNotNull())
    .groupBy("dni_limpio")
    .agg(
        F.countDistinct("id_empleado").alias("empleados_distintos"),
        F.count("*").alias("total_eventos")
    )
    .filter(F.col("empleados_distintos") > 1)
)

# Mostramos solo los identificadores de empleados;
# no exponemos el DNI completo en la evidencia.
display(
    df_dni_compartidos.agg(
        F.count("*").alias("dni_compartidos"),
        F.sum("empleados_distintos").alias("asociaciones_empleado_dni")
    )
)

print("=== DUPLICADOS POR LLAVE DE EVENTO ===")

df_duplicados = (
    df_bronze
    .groupBy(
        "id_empleado",
        "fecha_evento",
        "fecha_registro",
        "tipo_evento"
    )
    .count()
    .filter(F.col("count") > 1)
)

display(
    df_duplicados.agg(
        F.count("*").alias("llaves_evento_duplicadas"),
        F.coalesce(
            F.sum(F.col("count") - 1),
            F.lit(0)
        ).alias("filas_duplicadas_adicionales")
    )
)

In [0]:
from pyspark.sql import functions as F

df_calidad = (
    df_bronze
    .withColumn(
        "fecha_evento_valida",
        F.coalesce(
            F.try_to_timestamp(
                "fecha_evento", F.lit("yyyy-MM-dd")
            ).cast("date"),
            F.try_to_timestamp(
                "fecha_evento", F.lit("dd/MM/yyyy")
            ).cast("date")
        )
    )
    .withColumn(
        "salario_valido",
        F.expr(
            "try_cast("
            "regexp_replace(trim(salario), ',', '.') "
            "AS DECIMAL(12,2))"
        )
    )
    .withColumn(
        "rechazo_fecha",
        F.col("fecha_evento_valida").isNull()
        | F.col("id_empleado").isNull()
        | (F.trim("id_empleado") == "")
    )
    .withColumn(
        "rechazo_salario",
        F.col("salario_valido").isNull()
        | (~F.col("salario_valido").between(500, 20000))
    )
    .withColumn(
        "rechazo_dni",
        F.col("dni").isNull()
        | (F.trim("dni") == "")
    )
    .withColumn(
        "rechazado",
        F.col("rechazo_fecha")
        | F.col("rechazo_salario")
        | F.col("rechazo_dni")
    )
)

display(
    df_calidad.agg(
        F.count("*").alias("total_bronze"),
        F.sum(F.col("rechazo_fecha").cast("int"))
            .alias("rechazo_fecha"),
        F.sum(F.col("rechazo_salario").cast("int"))
            .alias("rechazo_salario"),
        F.sum(F.col("rechazo_dni").cast("int"))
            .alias("rechazo_dni"),
        F.sum(F.col("rechazado").cast("int"))
            .alias("total_rechazados"),
        F.sum((~F.col("rechazado")).cast("int"))
            .alias("total_aceptados")
    )
)

In [0]:
from pyspark.sql import functions as F

df_eventos_validos = (
    df_calidad
    .filter(~F.col("rechazado"))
    .withColumn(
        "tipo_evento_normalizado",
        F.when(
            F.lower(F.trim("tipo_evento")).isin("alta"),
            "alta"
        )
        .when(
            F.lower(F.trim("tipo_evento")).isin("transferencia"),
            "transferencia"
        )
        .when(
            F.lower(F.trim("tipo_evento")).isin(
                "cambio_salario", "cambio salario"
            ),
            "cambio_salario"
        )
        .when(
            F.lower(F.trim("tipo_evento")).isin("baja", "cese"),
            "baja"
        )
    )
    .withColumn(
        "fecha_registro_ts",
        F.try_to_timestamp(
            "fecha_registro",
            F.lit("yyyy-MM-dd HH:mm:ss")
        )
    )
    .withColumn(
        "dni_limpio",
        F.trim("dni")
    )
    .withColumn(
        "id_empleado_limpio",
        F.trim("id_empleado")
    )
)

print("=== EVENTOS NORMALIZADOS ===")

display(
    df_eventos_validos
    .groupBy("tipo_evento_normalizado")
    .count()
    .orderBy("tipo_evento_normalizado")
)

print("=== VALIDACIÓN DE TRANSFORMACIONES ===")

display(
    df_eventos_validos.agg(
        F.count("*").alias("total_eventos_validos"),
        F.sum(
            F.col("tipo_evento_normalizado")
            .isNull().cast("int")
        ).alias("eventos_sin_normalizar"),
        F.sum(
            F.col("fecha_registro_ts")
            .isNull().cast("int")
        ).alias("fechas_registro_invalidas"),
        F.countDistinct("id_empleado_limpio")
            .alias("empleados_distintos")
    )
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Empleados con llegada tardía y eventos del mismo día
empleados_revision = [
    "EMP-00017",  # Llegada tardía
    "EMP-00039",  # Llegada tardía
    "EMP-00010",  # Dos eventos el mismo día
    "EMP-00352"   # Dos eventos el mismo día
]

w_historia = (
    Window.partitionBy("id_empleado_limpio")
    .orderBy(
        F.col("fecha_evento_valida").asc(),
        F.col("fecha_registro_ts").asc(),
        F.col("archivo_origen").asc()
    )
)

df_secuencia = (
    df_eventos_validos
    .filter(
        F.col("id_empleado_limpio").isin(empleados_revision)
    )
    .withColumn(
        "orden_cronologico",
        F.row_number().over(w_historia)
    )
    .withColumn(
        "siguiente_fecha_evento",
        F.lead("fecha_evento_valida").over(w_historia)
    )
    .select(
        "id_empleado_limpio",
        "orden_cronologico",
        "tipo_evento_normalizado",
        "fecha_evento_valida",
        "fecha_registro_ts",
        "siguiente_fecha_evento",
        "sucursal_id",
        "cargo",
        "salario_valido",
        "archivo_origen"
    )
)

display(
    df_secuencia.orderBy(
        "id_empleado_limpio",
        "orden_cronologico"
    )
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Orden determinista de eventos dentro del mismo día
w_ultimo_evento_dia = (
    Window
    .partitionBy(
        "id_empleado_limpio",
        "fecha_evento_valida"
    )
    .orderBy(
        F.col("fecha_registro_ts").desc(),
        F.col("archivo_origen").desc(),
        F.col("tipo_evento_normalizado").desc()
    )
)

# Conservamos el último evento efectivo de cada día
df_estado_diario = (
    df_eventos_validos
    .withColumn(
        "orden_dia",
        F.row_number().over(w_ultimo_evento_dia)
    )
    .filter(F.col("orden_dia") == 1)
    .withColumn(
        "activo",
        F.col("tipo_evento_normalizado") != "baja"
    )
)

print("=== RESUMEN DE EVENTOS Y ESTADOS DIARIOS ===")

display(
    df_estado_diario.agg(
        F.count("*").alias("versiones_diarias"),
        F.countDistinct("id_empleado_limpio").alias("empleados_distintos"),
        F.sum(F.col("activo").cast("int")).alias("versiones_activas"),
        F.sum((~F.col("activo")).cast("int")).alias("versiones_baja")
    )
)

print("=== EVENTOS MÚLTIPLES DEL MISMO DÍA ===")

display(
    df_estado_diario
    .filter(
        F.col("id_empleado_limpio").isin(
            "EMP-00010", "EMP-00352"
        )
    )
    .select(
        "id_empleado_limpio",
        "fecha_evento_valida",
        "fecha_registro_ts",
        "tipo_evento_normalizado",
        "sucursal_id",
        "activo"
    )
    .orderBy(
        "id_empleado_limpio",
        "fecha_evento_valida"
    )
)

In [0]:
from pyspark.sql import functions as F

df_atributos_eventos = (
    df_eventos_validos
    .groupBy("tipo_evento_normalizado")
    .agg(
        F.count("*").alias("total_eventos"),

        F.sum(
            F.col("nombre").isNull().cast("int")
        ).alias("nombre_nulo"),

        F.sum(
            F.col("sucursal_id").isNull().cast("int")
        ).alias("sucursal_nula"),

        F.sum(
            F.col("cargo").isNull().cast("int")
        ).alias("cargo_nulo"),

        F.sum(
            F.col("salario_valido").isNull().cast("int")
        ).alias("salario_nulo"),

        F.sum(
            (
                (F.trim(F.col("sucursal_id")) == "") |
                (F.trim(F.col("cargo")) == "")
            ).cast("int")
        ).alias("atributos_vacios")
    )
    .orderBy("tipo_evento_normalizado")
)

print("=== COMPLETITUD DE ATRIBUTOS POR EVENTO ===")
display(df_atributos_eventos)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

w_inicio = (
    Window
    .partitionBy("id_empleado_limpio")
    .orderBy(
        F.col("fecha_evento_valida").asc(),
        F.col("fecha_registro_ts").asc(),
        F.col("archivo_origen").asc()
    )
)

df_primer_evento = (
    df_eventos_validos
    .withColumn(
        "orden_evento",
        F.row_number().over(w_inicio)
    )
    .filter(F.col("orden_evento") == 1)
)

print("=== PRIMER EVENTO VÁLIDO POR EMPLEADO ===")

display(
    df_primer_evento
    .groupBy("tipo_evento_normalizado")
    .agg(
        F.count("*").alias("total_empleados")
    )
    .orderBy("tipo_evento_normalizado")
)

print("=== EMPLEADOS SIN ALTA COMO PRIMER EVENTO ===")

df_sin_alta_inicial = (
    df_primer_evento
    .filter(F.col("tipo_evento_normalizado") != "alta")
)

display(
    df_sin_alta_inicial.agg(
        F.count("*").alias("empleados_sin_alta_inicial"),
        F.countDistinct("id_empleado_limpio")
            .alias("empleados_distintos")
    )
)

print("=== EJEMPLOS SIN ALTA INICIAL ===")

display(
    df_sin_alta_inicial
    .select(
        "id_empleado_limpio",
        "tipo_evento_normalizado",
        "fecha_evento_valida",
        "fecha_registro_ts",
        "archivo_origen"
    )
    .orderBy("id_empleado_limpio")
    .limit(15)
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Los 36 empleados identificados en la celda 14
df_empleados_revision = (
    df_sin_alta_inicial
    .select(
        F.col("id_empleado_limpio").alias("id_revision")
    )
    .distinct()
)

# Buscar todos sus eventos originales en Bronze
df_eventos_revision = (
    df_calidad
    .withColumn(
        "id_empleado_limpio",
        F.trim(F.col("id_empleado"))
    )
    .join(
        df_empleados_revision,
        F.col("id_empleado_limpio") == F.col("id_revision"),
        "inner"
    )
    .withColumn(
        "tipo_evento_revision",
        F.lower(
            F.regexp_replace(
                F.trim(F.col("tipo_evento")),
                " ",
                "_"
            )
        )
    )
    .withColumn(
        "motivo_rechazo",
        F.concat_ws(
            ", ",
            F.when(F.col("rechazo_fecha"), "FECHA_INVALIDA"),
            F.when(F.col("rechazo_salario"), "SALARIO_INVALIDO"),
            F.when(F.col("rechazo_dni"), "DNI_VACIO")
        )
    )
)

print("=== RESUMEN DE EVENTOS DE LOS 36 EMPLEADOS ===")

display(
    df_eventos_revision.agg(
        F.count("*").alias("total_eventos_bronze"),
        F.countDistinct("id_empleado_limpio")
            .alias("empleados_revisados"),
        F.sum(F.col("rechazado").cast("int"))
            .alias("eventos_rechazados"),
        F.sum((~F.col("rechazado")).cast("int"))
            .alias("eventos_aceptados"),
        F.sum(
            (F.col("tipo_evento_revision") == "alta").cast("int")
        ).alias("altas_en_bronze"),
        F.sum(
            (
                (F.col("tipo_evento_revision") == "alta")
                & F.col("rechazado")
            ).cast("int")
        ).alias("altas_rechazadas")
    )
)

print("=== ALTAS RECHAZADAS Y SUS MOTIVOS ===")

display(
    df_eventos_revision
    .filter(F.col("tipo_evento_revision") == "alta")
    .groupBy("motivo_rechazo")
    .agg(
        F.count("*").alias("total_altas"),
        F.countDistinct("id_empleado_limpio")
            .alias("empleados_afectados")
    )
    .orderBy(F.desc("total_altas"))
)

In [0]:
from pyspark.sql import functions as F

# Identificar los 36 empleados con alta inicial rechazada
df_ids_cuarentena = (
    df_empleados_revision
    .select(
        F.col("id_revision").alias("id_empleado_bloqueado")
    )
    .distinct()
    .withColumn("bloqueo_historial", F.lit(True))
)

# Aplicar las reglas individuales y el bloqueo por empleado
df_calidad_final = (
    df_calidad
    .withColumn(
        "id_empleado_limpio",
        F.trim(F.col("id_empleado"))
    )
    .join(
        df_ids_cuarentena,
        F.col("id_empleado_limpio")
        == F.col("id_empleado_bloqueado"),
        "left"
    )
    .withColumn(
        "rechazo_historial",
        F.coalesce(
            F.col("bloqueo_historial"),
            F.lit(False)
        )
    )
    .withColumn(
        "rechazado_final",
        F.col("rechazado") |
        F.col("rechazo_historial")
    )
    .withColumn(
        "motivo_rechazo_final",
        F.concat_ws(
            ", ",
            F.when(
                F.col("rechazo_fecha"),
                "FECHA_INVALIDA"
            ),
            F.when(
                F.col("rechazo_salario"),
                "SALARIO_INVALIDO"
            ),
            F.when(
                F.col("rechazo_dni"),
                "DNI_VACIO"
            ),
            F.when(
                F.col("rechazo_historial"),
                "HISTORIAL_INCOMPLETO_ALTA_RECHAZADA"
            )
        )
    )
)

print("=== CONCILIACION FINAL CON CUARENTENA POR EMPLEADO ===")

display(
    df_calidad_final.agg(
        F.count("*").alias("total_bronze"),

        F.sum(
            F.col("rechazado_final").cast("int")
        ).alias("total_cuarentena"),

        F.sum(
            (~F.col("rechazado_final")).cast("int")
        ).alias("total_aceptados"),

        F.sum(
            F.col("rechazo_historial").cast("int")
        ).alias("eventos_empleados_bloqueados"),

        F.countDistinct(
            F.when(
                F.col("rechazo_historial"),
                F.col("id_empleado_limpio")
            )
        ).alias("empleados_bloqueados")
    )
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Eventos que superaron todas las reglas de calidad
df_base_scd2 = (
    df_calidad_final
    .filter(~F.col("rechazado_final"))
    .withColumn("id_empleado_scd", F.trim("id_empleado"))
    .withColumn(
        "tipo_evento_scd",
        F.when(
            F.lower(F.trim("tipo_evento")) == "alta", "alta"
        ).when(
            F.lower(F.trim("tipo_evento")) == "transferencia",
            "transferencia"
        ).when(
            F.lower(
                F.regexp_replace(F.trim("tipo_evento"), " ", "_")
            ) == "cambio_salario",
            "cambio_salario"
        ).when(
            F.lower(F.trim("tipo_evento")).isin("baja", "cese"),
            "baja"
        )
    )
    .withColumn(
        "fecha_registro_scd",
        F.try_to_timestamp(
            "fecha_registro",
            F.lit("yyyy-MM-dd HH:mm:ss")
        )
    )
)

w_secuencia = (
    Window.partitionBy("id_empleado_scd")
    .orderBy(
        F.col("fecha_evento_valida"),
        F.col("fecha_registro_scd"),
        F.col("archivo_origen")
    )
)

df_control_secuencia = (
    df_base_scd2
    .withColumn(
        "numero_evento",
        F.row_number().over(w_secuencia)
    )
    .withColumn(
        "evento_anterior",
        F.lag("tipo_evento_scd").over(w_secuencia)
    )
    .withColumn(
        "primer_evento",
        F.first("tipo_evento_scd").over(
            w_secuencia.rowsBetween(
                Window.unboundedPreceding,
                Window.unboundedFollowing
            )
        )
    )
)

display(
    df_control_secuencia.agg(
        F.count("*").alias("eventos_aceptados"),

        F.countDistinct("id_empleado_scd")
            .alias("empleados_distintos"),

        F.countDistinct(
            F.when(
                (F.col("numero_evento") == 1)
                & (F.col("tipo_evento_scd") != "alta"),
                F.col("id_empleado_scd")
            )
        ).alias("empleados_sin_alta_inicial"),

        F.sum(
            (
                (F.col("evento_anterior") == "baja")
                & (~F.col("tipo_evento_scd").isin("alta"))
            ).cast("int")
        ).alias("eventos_despues_baja"),

        F.sum(
            F.col("tipo_evento_scd").isNull().cast("int")
        ).alias("eventos_desconocidos")
    )
)

In [0]:
# Comprobar que no existen anomalías que bloqueen SCD2
controles = df_control_secuencia.agg(
    F.countDistinct(
        F.when(
            (F.col("numero_evento") == 1)
            & (F.col("tipo_evento_scd") != "alta"),
            F.col("id_empleado_scd")
        )
    ).alias("sin_alta"),
    F.sum(
        (
            (F.col("evento_anterior") == "baja")
            & (~F.col("tipo_evento_scd").isin("alta"))
        ).cast("int")
    ).alias("post_baja"),
    F.sum(
        F.col("tipo_evento_scd").isNull().cast("int")
    ).alias("desconocidos")
).first()

if any((controles[x] or 0) > 0
       for x in ["sin_alta", "post_baja", "desconocidos"]):
    raise ValueError(
        "Existen secuencias inconsistentes. "
        "Revisar la celda 17 antes de construir SCD2."
    )

# Último evento válido de cada empleado en cada día
w_dia = (
    Window.partitionBy(
        "id_empleado_scd",
        "fecha_evento_valida"
    )
    .orderBy(
        F.col("fecha_registro_scd").desc(),
        F.col("archivo_origen").desc(),
        F.col("tipo_evento_scd").desc()
    )
)

df_ultimo_dia = (
    df_base_scd2
    .withColumn("orden_dia", F.row_number().over(w_dia))
    .filter(F.col("orden_dia") == 1)
)

# Orden cronológico de las versiones
w_version = (
    Window.partitionBy("id_empleado_scd")
    .orderBy("fecha_evento_valida")
)

df_scd2 = (
    df_ultimo_dia
    .withColumn(
        "version",
        F.row_number().over(w_version)
    )
    .withColumn(
        "vigente_desde",
        F.col("fecha_evento_valida")
    )
    .withColumn(
        "vigente_hasta",
        F.lead("fecha_evento_valida").over(w_version)
    )
    .withColumn(
        "es_actual",
        F.col("vigente_hasta").isNull()
    )
    .withColumn(
        "activo",
        F.col("tipo_evento_scd") != "baja"
    )
    .select(
        F.col("id_empleado_scd").alias("id_empleado"),
        F.trim("nombre").alias("nombre"),
        F.trim("dni").alias("dni"),
        F.trim("email").alias("email"),
        F.col("salario_valido").alias("salario"),
        F.trim("sucursal_id").alias("sucursal_id"),
        F.trim("cargo").alias("cargo"),
        F.col("tipo_evento_scd").alias("tipo_evento"),
        F.col("fecha_evento_valida").alias("fecha_evento"),
        F.col("fecha_registro_scd").alias("fecha_registro"),
        "version",
        "vigente_desde",
        "vigente_hasta",
        "es_actual",
        "activo"
    )
)

print("=== RESUMEN DE VERSIONES SCD2 ===")

display(
    df_scd2.agg(
        F.count("*").alias("total_versiones"),
        F.countDistinct("id_empleado").alias("empleados"),
        F.sum(F.col("es_actual").cast("int"))
            .alias("versiones_actuales"),
        F.sum(
            (F.col("es_actual") & F.col("activo")).cast("int")
        ).alias("empleados_activos_actuales"),
        F.sum(
            (F.col("es_actual") & ~F.col("activo")).cast("int")
        ).alias("empleados_inactivos_actuales")
    )
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

w_validacion = (
    Window.partitionBy("id_empleado")
    .orderBy("vigente_desde", "version")
)

df_validacion_scd2 = (
    df_scd2
    .withColumn(
        "fin_version_anterior",
        F.lag("vigente_hasta").over(w_validacion)
    )
    .withColumn(
        "version_anterior",
        F.lag("version").over(w_validacion)
    )
    .withColumn(
        "intervalo_invalido",
        F.col("vigente_hasta").isNotNull()
        & (F.col("vigente_hasta") <= F.col("vigente_desde"))
    )
    .withColumn(
        "vigencia_superpuesta",
        F.col("version_anterior").isNotNull()
        & F.col("fin_version_anterior").isNotNull()
        & (F.col("vigente_desde") < F.col("fin_version_anterior"))
    )
    .withColumn(
        "version_discontinua",
        F.col("version_anterior").isNotNull()
        & F.col("fin_version_anterior").isNotNull()
        & (F.col("vigente_desde") != F.col("fin_version_anterior"))
    )
)

df_actuales = (
    df_scd2
    .groupBy("id_empleado")
    .agg(
        F.sum(F.col("es_actual").cast("int"))
        .alias("total_actuales")
    )
)

print("=== CONTROL DE INTEGRIDAD SCD TIPO 2 ===")

display(
    df_validacion_scd2.agg(
        F.count("*").alias("total_versiones"),
        F.sum(F.col("intervalo_invalido").cast("int"))
            .alias("intervalos_invalidos"),
        F.sum(F.col("vigencia_superpuesta").cast("int"))
            .alias("vigencias_superpuestas"),
        F.sum(F.col("version_discontinua").cast("int"))
            .alias("vigencias_discontinuas"),
        F.sum(
            F.col("vigente_desde").isNull().cast("int")
        ).alias("inicios_nulos")
    )
)

print("=== CONTROL DE VERSION ACTUAL POR EMPLEADO ===")

display(
    df_actuales.agg(
        F.count("*").alias("empleados"),
        F.sum(
            (F.col("total_actuales") != 1).cast("int")
        ).alias("empleados_con_actual_incorrecto"),
        F.max("total_actuales").alias("max_versiones_actuales")
    )
)

print("=== EJEMPLO HISTORICO SIN DATOS SENSIBLES ===")

display(
    df_scd2
    .filter(F.col("id_empleado") == "EMP-00017")
    .select(
        "id_empleado",
        "version",
        "tipo_evento",
        "sucursal_id",
        "vigente_desde",
        "vigente_hasta",
        "es_actual",
        "activo"
    )
    .orderBy("version")
)

In [0]:
spark.sql("""
CREATE FUNCTION IF NOT EXISTS proyecto_final.silver.mask_dni(valor STRING)
RETURNS STRING
RETURN CASE
    WHEN is_account_group_member('electrocasa_ingenieria')
        THEN valor
    ELSE '********'
END
""")

spark.sql("""
CREATE FUNCTION IF NOT EXISTS proyecto_final.silver.mask_salario(
    valor DECIMAL(12,2)
)
RETURNS DECIMAL(12,2)
RETURN CASE
    WHEN is_account_group_member('electrocasa_ingenieria')
        THEN valor
    ELSE CAST(NULL AS DECIMAL(12,2))
END
""")

print("Funciones de enmascaramiento creadas.")

In [0]:
%sql
SELECT
    current_user() AS usuario_actual,
    is_account_group_member('electrocasa_ingenieria') AS pertenece_ingenieria;

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS proyecto_final.silver.empleados (
    id_empleado STRING NOT NULL,
    nombre STRING,
    dni STRING MASK proyecto_final.silver.mask_dni,
    email STRING,
    salario DECIMAL(12,2) MASK proyecto_final.silver.mask_salario,
    sucursal_id STRING,
    cargo STRING,
    tipo_evento STRING,
    fecha_evento DATE,
    fecha_registro TIMESTAMP,
    version INT,
    vigente_desde DATE,
    vigente_hasta DATE,
    es_actual BOOLEAN,
    activo BOOLEAN
)
USING DELTA
COMMENT 'Historial SCD Tipo 2 de empleados de ElectroCasa'
""")

print("Tabla Silver Empleados creada.")

In [0]:
df_scd2.createOrReplaceTempView("vw_empleados_scd2")

spark.sql("""
INSERT OVERWRITE proyecto_final.silver.empleados
SELECT
    id_empleado,
    nombre,
    dni,
    email,
    salario,
    sucursal_id,
    cargo,
    tipo_evento,
    fecha_evento,
    fecha_registro,
    version,
    vigente_desde,
    vigente_hasta,
    es_actual,
    activo
FROM vw_empleados_scd2
""")

print("Datos SCD Tipo 2 cargados correctamente.")

In [0]:
%sql
SELECT
    COUNT(*) AS total_versiones,
    COUNT(DISTINCT id_empleado) AS empleados,
    SUM(CASE WHEN es_actual THEN 1 ELSE 0 END) AS versiones_actuales,
    SUM(CASE WHEN es_actual AND activo THEN 1 ELSE 0 END) AS activos,
    SUM(CASE WHEN es_actual AND NOT activo THEN 1 ELSE 0 END) AS inactivos
FROM proyecto_final.silver.empleados;

In [0]:
%sql
SELECT
    id_empleado,
    nombre,
    dni,
    email,
    salario,
    sucursal_id,
    cargo,
    tipo_evento,
    fecha_evento,
    fecha_registro,
    version,
    vigente_desde,
    vigente_hasta,
    es_actual,
    activo
FROM proyecto_final.silver.empleados
ORDER BY id_empleado, version
LIMIT 100;

In [0]:
from pyspark.sql import functions as F

# Columnas originales de Bronze para auditoría
columnas_originales = df_bronze.columns

df_cuarentena_empleados = (
    df_calidad_final
    .filter(F.col("rechazado_final"))
    .select(
        F.lit("empleados").alias("fuente"),

        F.concat(
            F.coalesce(
                F.col("id_empleado_limpio"),
                F.lit("SIN_ID")
            ),
            F.lit("_"),
            F.sha2(
                F.to_json(
                    F.struct(*[
                        F.col(c).alias(c)
                        for c in columnas_originales
                    ])
                ),
                256
            )
        ).alias("clave_registro"),

        F.col("motivo_rechazo_final").alias("motivo_rechazo"),

        F.current_timestamp().alias("fecha_rechazo"),

        F.to_json(
            F.struct(*[
                F.col(c).alias(c)
                for c in columnas_originales
            ])
        ).alias("datos_originales"),

        F.col("archivo_origen")
    )
)

print("Registros preparados:", df_cuarentena_empleados.count())

assert df_cuarentena_empleados.count() == 181, \
    "El total de rechazos no coincide con la conciliación."

print("Validación correcta: 181 registros.")

In [0]:
(
    df_cuarentena_empleados.write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere", "fuente = 'empleados'")
    .saveAsTable("proyecto_final.silver.cuarentena")
)

print("Cuarentena de empleados cargada correctamente.")

In [0]:
%sql
SELECT
    fuente,
    COUNT(*) AS total_rechazados,
    COUNT(DISTINCT clave_registro) AS registros_distintos,
    COUNT(DISTINCT archivo_origen) AS archivos_origen
FROM proyecto_final.silver.cuarentena
WHERE fuente = 'empleados'
GROUP BY fuente;

In [0]:
%sql
SELECT
    motivo_rechazo,
    COUNT(*) AS total
FROM proyecto_final.silver.cuarentena
WHERE fuente = 'empleados'
GROUP BY motivo_rechazo
ORDER BY total DESC;

In [0]:
%sql
SELECT
    fuente,
    clave_registro,
    motivo_rechazo,
    fecha_rechazo,
    archivo_origen
FROM proyecto_final.silver.cuarentena
ORDER BY motivo_rechazo, clave_registro
